# LeetCode 2481 — Minimum Cuts to Divide a Circle bằng Python

Link đề: https://leetcode.com/problems/minimum-cuts-to-divide-a-circle/ · Độ khó: **Easy** · Chủ đề: Math, Geometry

Skeleton LeetCode:

```python
class Solution:
    def numberOfCuts(self, n: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu thế nào là một **nhát cắt hợp lệ** của đề (đoạn thẳng đi qua tâm hình tròn) và vì sao mọi nhát cắt đều là một **đường kính** hoặc một **bán kính**.
2. Biết biến bài hình học thành bài **đếm tia**: chia thành `n` phần bằng nhau nghĩa là có `n` tia xuất phát từ tâm, cách đều nhau.
3. Rút ra công thức theo tính chẵn lẻ: `n = 1` cần `0` nhát, `n` chẵn cần `n/2` nhát, `n` lẻ lớn hơn 1 cần `n` nhát.
4. Viết 4 cách: duyệt số nhát `k` tăng dần, mô phỏng phủ tia đối diện, công thức rẽ nhánh và bảng tra cứu; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi quen thuộc: quên trường hợp `n = 1`, dùng `n // 2` cho cả số lẻ, dùng `(n + 1) // 2`, và nhầm `n` chẵn với `n` chia hết cho 4.

## 2. Đề bài & ví dụ

Một **nhát cắt hợp lệ** của hình tròn là một trong hai loại:

- Một đoạn thẳng đi qua **tâm** và có **hai đầu** nằm trên đường tròn (một đường kính).
- Một đoạn thẳng đi qua **tâm** và có **một đầu** nằm trên đường tròn (một bán kính).

Cho số nguyên `n`. Hãy trả về **số nhát cắt ít nhất** cần thực hiện để chia hình tròn thành `n` phần **bằng nhau**.

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `4` | `2` | hai đường kính vuông góc cho 4 phần bằng nhau |
| 2 | `3` | `3` | cần 3 bán kính cách nhau 120 độ; không có cách nào dùng ít hơn 3 nhát |

Constraints: `1 <= n <= 100`.

In [1]:
examples = [(4, 2), (3, 3)]
for n, expected in examples:
    print(f"n={n}: mong đợi {expected} nhát cắt")
# n = 1 chưa cần cắt: hình tròn nguyên vẹn đã là 1 phần
print("n=1 -> 0 nhát (không cần cắt)")
print("n=2 -> 1 nhát (một đường kính)")

n=4: mong đợi 2 nhát cắt
n=3: mong đợi 3 nhát cắt
n=1 -> 0 nhát (không cần cắt)
n=2 -> 1 nhát (một đường kính)


## 3. Kiến thức nền

### 3.1. Toán tử `%` (chia lấy dư) và tính chẵn lẻ

`n % 2 == 0` nghĩa là `n` chẵn, `n % 2 == 1` nghĩa là `n` lẻ. Toán tử `//` là chia nguyên (làm tròn xuống). Với `n` chẵn thì `n // 2` đúng bằng `n / 2`.

In [2]:
for n in (1, 2, 3, 4, 7, 10):
    print(n, "chẵn" if n % 2 == 0 else "lẻ", "| n // 2 =", n // 2, "| n % 2 =", n % 2)

1 lẻ | n // 2 = 0 | n % 2 = 1
2 chẵn | n // 2 = 1 | n % 2 = 0
3 lẻ | n // 2 = 1 | n % 2 = 1
4 chẵn | n // 2 = 2 | n % 2 = 0
7 lẻ | n // 2 = 3 | n % 2 = 1
10 chẵn | n // 2 = 5 | n % 2 = 0


### 3.2. Phép toán bit `n & 1`

`n & 1` lấy bit cuối của `n`: bằng `1` nếu `n` lẻ và `0` nếu `n` chẵn. Đây là cách viết ngắn của `n % 2`.

In [3]:
for n in range(1, 9):
    print(n, bin(n), "n & 1 =", n & 1, "| n % 2 =", n % 2)

1 0b1 n & 1 = 1 | n % 2 = 1
2 0b10 n & 1 = 0 | n % 2 = 0
3 0b11 n & 1 = 1 | n % 2 = 1
4 0b100 n & 1 = 0 | n % 2 = 0
5 0b101 n & 1 = 1 | n % 2 = 1
6 0b110 n & 1 = 0 | n % 2 = 0
7 0b111 n & 1 = 1 | n % 2 = 1
8 0b1000 n & 1 = 0 | n % 2 = 0


### 3.3. `set` để theo dõi tập các tia chưa được cắt

Ta sẽ mô tả mỗi tia bằng một chỉ số `0..n-1`. `set` cho phép `add`, `discard` (bỏ nếu có, không lỗi nếu không có) và kiểm tra `in` trong thời gian trung bình `O(1)`.

In [4]:
s = {0, 1, 2, 3}
s.discard(1)
s.discard(10)       # không lỗi dù 10 không có
print(s, 2 in s, 1 in s, len(s))

{0, 2, 3} True False 3


### 3.4. Phép `%` để đi vòng tròn (chỉ số quay vòng)

Tia thứ `i` trong `n` tia nằm ở góc `i * 360 / n` độ. Tia đối diện của tia `i` (xoay thêm 180 độ) là tia `(i + n/2) % n` nếu `n` chẵn. Phép `%` giúp quay vòng khi chỉ số vượt quá `n - 1`.

In [5]:
n = 6
for i in range(n):
    print(f"tia {i}: góc {i * 360 / n:.0f} độ, tia đối diện = {(i + n // 2) % n}")

tia 0: góc 0 độ, tia đối diện = 3
tia 1: góc 60 độ, tia đối diện = 4
tia 2: góc 120 độ, tia đối diện = 5
tia 3: góc 180 độ, tia đối diện = 0
tia 4: góc 240 độ, tia đối diện = 1
tia 5: góc 300 độ, tia đối diện = 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mọi nhát cắt đi qua tâm. Vì vậy các đường cắt trên hình tròn là các **tia từ tâm**. Muốn chia thành `n` phần **bằng nhau** ta cần đúng `n` tia cách đều nhau (khi `n >= 2`): các góc `0, 360/n, 2*360/n, ...`.

Tính tay vài trường hợp:

- `n = 1`: không cần tia nào, hình tròn đã là một phần. Đáp án `0`.
- `n = 2`: cần 2 tia đối nhau (góc 0 và 180). **Một** đường kính tạo ra cả hai tia. Đáp án `1`.
- `n = 4`: bốn tia ở 0, 90, 180, 270. Hai đường kính (0-180 và 90-270). Đáp án `2`.
- `n = 3`: ba tia ở 0, 120, 240. Không cặp tia nào đối nhau (180 độ) nên mỗi tia phải là một bán kính riêng. Đáp án `3`.
- `n = 6`: sáu tia ở 0, 60, ..., 300; tia 0 đối tia 3, tia 1 đối tia 4, tia 2 đối tia 5. Ba đường kính. Đáp án `3`.

In [6]:
for n in (1, 2, 3, 4, 5, 6):
    angles = [round(i * 360 / n, 1) for i in range(n)] if n > 1 else []
    print(f"n={n}: các tia ở góc {angles}")

n=1: các tia ở góc []
n=2: các tia ở góc [0.0, 180.0]
n=3: các tia ở góc [0.0, 120.0, 240.0]
n=4: các tia ở góc [0.0, 90.0, 180.0, 270.0]
n=5: các tia ở góc [0.0, 72.0, 144.0, 216.0, 288.0]
n=6: các tia ở góc [0.0, 60.0, 120.0, 180.0, 240.0, 300.0]


## 5. Cách 1 — Brute Force / baseline: thử số nhát `k` tăng dần

**Ý tưởng:** với `k` nhát cắt, ta biết **số phần bằng nhau lớn nhất có thể** theo hai kiểu cắt:

- `k` đường kính cách đều nhau cho `2k` phần bằng nhau (với `k >= 1`).
- `k` bán kính cách đều nhau cho `k` phần bằng nhau (với `k >= 2`).
- `k = 0` cho `1` phần.

Ta thử `k = 0, 1, 2, ...` và trả về `k` đầu tiên cho ra đúng `n` phần.

In [7]:
class SolutionBrute:
    def numberOfCuts(self, n: int) -> int:
        for k in range(0, n + 1):
            if k == 0 and n == 1:
                return k                  # không cắt: 1 phần
            if k >= 1 and 2 * k == n:
                return k                  # k đường kính -> 2k phần
            if k >= 2 and k == n:
                return k                  # k bán kính -> k phần
        return n

In [8]:
sol = SolutionBrute()
print([sol.numberOfCuts(n) for n in (1, 2, 3, 4, 5, 6, 7, 8)])
print(sol.numberOfCuts(4), sol.numberOfCuts(3))     # 2 3

[0, 1, 3, 2, 5, 3, 7, 4]
2 3


**Độ phức tạp:** Time `O(n)` (thử tối đa `n + 1` giá trị `k`), Space `O(1)`. **Nút thắt:** vòng lặp chỉ để tìm `k` mà ta có thể suy ra trực tiếp từ tính chẵn lẻ của `n`.

## 6. Ý tưởng thuật toán / bước đột phá

Nhìn lại bảng ở mục 4 ta thấy:

- Mỗi **đường kính** tạo ra **hai tia đối nhau**, chỉ có lợi khi hai tia đó đều cần (cùng thuộc `n` tia cách đều).
- Với `n` **chẵn**, tia `i` luôn có tia đối `i + n/2` cũng nằm trong tập `n` tia. Vậy ta ghép chúng thành `n / 2` cặp, mỗi cặp một đường kính: `n / 2` nhát.
- Với `n` **lẻ** (và `n >= 3`), tia `i` ở góc `i * 360 / n`; để tia đối nằm ở `i * 360 / n + 180` phải là bội của `360 / n`, tức `n / 2` phải nguyên, vô lý khi `n` lẻ. Vậy không có đường kính nào "hai đầu cùng dùng được", ta cần `n` bán kính: `n` nhát.
- Trường hợp riêng `n = 1`: `0` nhát.

**Quy tắc:** *nếu `n == 1` thì `0`; nếu `n` chẵn thì `n // 2`; ngược lại `n`.*

**Invariant:** số nhát cắt tối thiểu bằng số tia cần có trừ số cặp tia đối nhau được gộp chung một đường kính.

## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng "phủ tia" bằng `set`

Giữ tập `con_lai` gồm các tia chưa được cắt `{0, ..., n-1}`. Lấy một tia bất kỳ, cắt nó (một nhát). Nếu `n` chẵn, nhát cắt đó là đường kính nên **cũng xóa luôn tia đối diện**. Lặp đến khi hết tia. Với `n = 1` không có tia nào cần cắt.

In [9]:
class SolutionRays:
    def numberOfCuts(self, n: int) -> int:
        if n == 1:
            return 0                              # hình tròn nguyên vẹn
        con_lai = set(range(n))                   # các tia chưa được cắt
        cuts = 0
        while con_lai:
            i = min(con_lai)                      # lấy một tia bất kỳ
            con_lai.discard(i)
            if n % 2 == 0:
                con_lai.discard((i + n // 2) % n) # đường kính xóa luôn tia đối
            cuts += 1
        return cuts

In [10]:
print([SolutionRays().numberOfCuts(n) for n in (1, 2, 3, 4, 5, 6, 7, 8)])

[0, 1, 3, 2, 5, 3, 7, 4]


**Độ phức tạp:** Time `O(n)` (mỗi tia bị xóa đúng một lần; `min` trên `set` tốn `O(n)` nên thực tế là `O(n^2)` nhưng `n <= 100` nên không đáng kể), Space `O(n)` cho tập tia.

### Cách 3 — Công thức rẽ nhánh

Áp dụng trực tiếp quy tắc ở mục 6 bằng ba nhánh `if`.

In [11]:
class SolutionFormula:
    def numberOfCuts(self, n: int) -> int:
        if n == 1:
            return 0
        if n % 2 == 0:
            return n // 2
        return n

In [12]:
print([SolutionFormula().numberOfCuts(n) for n in (1, 2, 3, 4, 5, 6, 7, 8)])

[0, 1, 3, 2, 5, 3, 7, 4]


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### Cách 4 — Bảng tra cứu dựng sẵn

Vì `n <= 100`, ta dựng bảng `table[n]` một lần (bằng Cách 1) rồi trả về `table[n]`. Hữu ích khi có rất nhiều truy vấn.

In [13]:
class SolutionTable:
    TABLE = {m: SolutionBrute().numberOfCuts(m) for m in range(1, 101)}

    def numberOfCuts(self, n: int) -> int:
        return self.TABLE[n]

In [14]:
print([SolutionTable().numberOfCuts(n) for n in (1, 2, 3, 4, 5, 6, 7, 8)])
print(len(SolutionTable.TABLE), "giá trị n đã dựng sẵn")

[0, 1, 3, 2, 5, 3, 7, 4]
100 giá trị n đã dựng sẵn


**Độ phức tạp:** Time `O(1)` mỗi truy vấn (dựng bảng `O(100^2)` một lần), Space `O(100)`.

## 8. TRACE từng bước

Trace Cách 2 với `n = 6`: tập tia `{0,1,2,3,4,5}`.

| Vòng | Tia chọn `i` | Tia đối `(i+3)%6` | `con_lai` sau khi xóa | `cuts` |
|---|---|---|---|---|
| 1 | 0 | 3 | `{1,2,4,5}` | 1 |
| 2 | 1 | 4 | `{2,5}` | 2 |
| 3 | 2 | 5 | `{}` | 3 |

Kết quả `3`. Với `n = 5` (lẻ) không có tia đối nào bị xóa chung nên cần `5` vòng.

In [15]:
def trace_rays(n):
    print("n =", n)
    if n == 1:
        print("n = 1: không cần cắt -> 0")
        return 0
    con_lai = set(range(n))
    cuts = 0
    while con_lai:
        i = min(con_lai)
        con_lai.discard(i)
        doi = None
        if n % 2 == 0:
            doi = (i + n // 2) % n
            con_lai.discard(doi)
        cuts += 1
        print(f"  cắt tia {i}, tia đối = {doi}, còn lại = {sorted(con_lai)}, cuts = {cuts}")
    print("kết quả:", cuts)
    return cuts

trace_rays(6)
trace_rays(5)
trace_rays(1)

n = 6
  cắt tia 0, tia đối = 3, còn lại = [1, 2, 4, 5], cuts = 1
  cắt tia 1, tia đối = 4, còn lại = [2, 5], cuts = 2
  cắt tia 2, tia đối = 5, còn lại = [], cuts = 3
kết quả: 3
n = 5
  cắt tia 0, tia đối = None, còn lại = [1, 2, 3, 4], cuts = 1
  cắt tia 1, tia đối = None, còn lại = [2, 3, 4], cuts = 2
  cắt tia 2, tia đối = None, còn lại = [3, 4], cuts = 3
  cắt tia 3, tia đối = None, còn lại = [4], cuts = 4
  cắt tia 4, tia đối = None, còn lại = [], cuts = 5
kết quả: 5
n = 1
n = 1: không cần cắt -> 0


0

## 9. Code nộp LeetCode

Công thức rẽ nhánh `O(1)`:

In [16]:
class Solution:
    def numberOfCuts(self, n: int) -> int:
        if n == 1:
            return 0
        return n // 2 if n % 2 == 0 else n

In [17]:
print(Solution().numberOfCuts(4))    # 2
print(Solution().numberOfCuts(3))    # 3
print(Solution().numberOfCuts(1))    # 0

2
3
0


## 10. Vì sao thuật toán đúng

1. **Mọi nhát cắt là tia hoặc đường kính.** Vì đoạn cắt phải đi qua tâm, nên nó chỉ là một bán kính (một tia) hoặc một đường kính (hai tia đối nhau). Các phần của hình tròn được xác định hoàn toàn bởi các tia đã cắt.
2. **Chia đều cần đúng `n` tia cách đều** (với `n >= 2`): góc mỗi phần phải bằng nhau nên các tia ở `i * 360 / n` độ. Ít tia hơn thì ra ít hơn `n` phần; nhiều tia hơn thì ra nhiều hơn `n` phần.
3. **Số nhát = số tia - số cặp được gộp.** Mỗi nhát cắt tối đa phủ hai tia (khi là đường kính), nên số nhát tối thiểu là `ceil` của số tia chia cho số tia mỗi nhát phủ được, mà việc phủ hai tia chỉ khả thi với cặp tia đối nhau.
4. **`n` chẵn:** tia `i` và `i + n/2` lệch đúng 180 độ. Ghép thành `n/2` cặp rời nhau nên `n/2` nhát, và không thể ít hơn vì mỗi nhát phủ tối đa 2 tia, cần phủ `n` tia.
5. **`n` lẻ (>= 3):** không cặp tia nào lệch 180 độ (vì `180 / (360/n) = n/2` không nguyên) nên mỗi nhát chỉ phủ được 1 tia: `n` nhát.
6. **`n = 1`:** hình tròn đã là một phần, không cần cắt.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử `k` tăng dần | `O(n)` | `O(1)` |
| 2. Phủ tia bằng `set` | `O(n^2)` do `min` (hoặc `O(n)` nếu duyệt theo chỉ số) | `O(n)` |
| 3. Công thức | `O(1)` | `O(1)` |
| 4. Bảng tra cứu | `O(1)` mỗi truy vấn | `O(100)` |

`n` là số phần cần chia. Vì `n <= 100`, mọi cách đều chạy tức thì; công thức là lựa chọn gọn nhất.

## 12. Bộ test

### 12.1. Ví dụ của đề, edge case và phản ví dụ

Gồm `n = 1` (không cắt), `n = 2` (nhỏ nhất chẵn), số lẻ nhỏ, số chẵn là bội của 4 và không phải bội của 4, và hai biên `99`, `100`.

In [18]:
sols = [SolutionBrute(), SolutionRays(), SolutionFormula(), SolutionTable(), Solution()]
cases = [
    (4, 2),      # ví dụ 1
    (3, 3),      # ví dụ 2
    (1, 0),      # không cắt
    (2, 1),      # một đường kính
    (5, 5),      # lẻ nhỏ
    (6, 3),      # chẵn, không chia hết cho 4
    (8, 4),      # chẵn, bội của 4
    (10, 5),     # chẵn, không chia hết cho 4
    (99, 99),    # lẻ lớn
    (100, 50),   # biên trên, chẵn
]
for n, want in cases:
    for s in sols:
        got = s.numberOfCuts(n)
        assert got == want, (type(s).__name__, n, got, want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 10


Phản ví dụ nổi tiếng: `n = 3`. Nhiều người nghĩ `3 // 2 + 1 = 2` hoặc `1` nhát; thực tế 2 nhát chỉ chia được thành 3 phần **không bằng nhau**.

In [19]:
assert Solution().numberOfCuts(3) == 3
assert Solution().numberOfCuts(3) != 3 // 2
assert Solution().numberOfCuts(1) == 0 and Solution().numberOfCuts(1) != 1
print("phản ví dụ n=3 và n=1 OK")

phản ví dụ n=3 và n=1 OK


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh `n` ngẫu nhiên trong khoảng cho phép `1 <= n <= 100` và so sánh cả 5 cách. Ngoài ra kiểm tra toàn bộ `n = 1..100` vì miền rất nhỏ.

In [20]:
import random
random.seed(2481)

for _ in range(2000):
    n = random.randint(1, 100)
    ans = [s.numberOfCuts(n) for s in sols]
    assert len(set(ans)) == 1, (n, ans)

for n in range(1, 101):
    ans = [s.numberOfCuts(n) for s in sols]
    assert len(set(ans)) == 1, (n, ans)
print("2000 test ngẫu nhiên + toàn bộ n = 1..100: 5 cách khớp nhau")

2000 test ngẫu nhiên + toàn bộ n = 1..100: 5 cách khớp nhau


### 12.3. Tính chất của đáp án

Với mọi `n` hợp lệ: đáp án nằm trong `[0, n]`; nếu `n >= 2` thì đáp án `>= n/2` (mỗi nhát phủ tối đa 2 tia); đáp án chỉ bằng `n/2` khi `n` chẵn.

In [21]:
for n in range(1, 101):
    ans = Solution().numberOfCuts(n)
    assert 0 <= ans <= n
    if n >= 2:
        assert 2 * ans >= n                # mỗi nhát phủ tối đa 2 tia
        assert (2 * ans == n) == (n % 2 == 0)
    else:
        assert ans == 0
print("các tính chất của đáp án đều đúng cho n = 1..100")

các tính chất của đáp án đều đúng cho n = 1..100


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên trường hợp `n = 1`

In [22]:
def sai_n_bang_1(n):
    return n // 2 if n % 2 == 0 else n          # SAI khi n = 1: trả về 1

print("n=1 đúng:", Solution().numberOfCuts(1), "| sai:", sai_n_bang_1(1))
assert Solution().numberOfCuts(1) != sai_n_bang_1(1)

n=1 đúng: 0 | sai: 1


**Vì sao sai:** `n = 1` là số lẻ nên nhánh "lẻ" trả `1`, nhưng hình tròn nguyên vẹn đã là một phần, đáp án là `0`. Luôn xử lý `n = 1` trước tiên.

### Lỗi 2: Dùng `n // 2` cho cả số lẻ

In [23]:
def sai_chia_hai(n):
    if n == 1:
        return 0
    return n // 2                               # SAI với n lẻ

print("n=3 đúng:", Solution().numberOfCuts(3), "| sai:", sai_chia_hai(3))
print("n=5 đúng:", Solution().numberOfCuts(5), "| sai:", sai_chia_hai(5))
assert Solution().numberOfCuts(3) != sai_chia_hai(3)
assert Solution().numberOfCuts(5) != sai_chia_hai(5)

n=3 đúng: 3 | sai: 1
n=5 đúng: 5 | sai: 2


**Vì sao sai:** `n // 2` chỉ đúng khi `n` chẵn vì ta ghép được các tia đối nhau. Với `n = 3`, hai nhát cắt đi qua tâm chỉ cho 3 phần **không bằng nhau**.

### Lỗi 3: Làm tròn lên `(n + 1) // 2`

In [24]:
def sai_lam_tron_len(n):
    if n == 1:
        return 0
    return (n + 1) // 2                         # SAI: n lẻ vẫn cần n nhát

print("n=3 đúng:", Solution().numberOfCuts(3), "| sai:", sai_lam_tron_len(3))
assert Solution().numberOfCuts(3) != sai_lam_tron_len(3)
assert Solution().numberOfCuts(7) != sai_lam_tron_len(7)

n=3 đúng: 3 | sai: 2


**Vì sao sai:** làm tròn lên cho `(3+1)//2 = 2` nhát, nhưng 2 nhát chỉ phủ tối đa 4 tia nên không thể cho đúng 3 phần bằng nhau. Với `n` lẻ không ghép được cặp nào.

### Lỗi 4: Nhầm "chẵn" với "chia hết cho 4"

Nhiều người thấy `n = 4 -> 2` và vội kết luận chỉ `n` chia hết cho 4 mới đủ điều kiện ghép đường kính.

In [25]:
def sai_chia_het_4(n):
    if n == 1:
        return 0
    return n // 2 if n % 4 == 0 else n          # SAI: n = 6 vẫn ghép được

print("n=6 đúng:", Solution().numberOfCuts(6), "| sai:", sai_chia_het_4(6))
assert Solution().numberOfCuts(6) != sai_chia_het_4(6)

n=6 đúng: 3 | sai: 6


**Vì sao sai:** chỉ cần `n` **chẵn** là có tia đối `(i + n/2) % n`. Với `n = 6` có ba đường kính tạo góc 60 độ, không cần chia hết cho 4.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài hình học nhưng đáp án chỉ phụ thuộc **tính chẵn lẻ**: hãy tính tay `n = 1..6` để thấy quy luật.
- Thấy "đường thẳng đi qua tâm" thì nghĩ ngay đến **cặp tia đối nhau** (cách nhau 180 độ).
- Trường hợp `n = 1` luôn là bẫy đầu tiên cần kiểm tra.
- Công thức ngắn gọn: `0 if n == 1 else (n // 2 if n % 2 == 0 else n)`.
- Dạng ngắn hơn nữa: `n if n & 1 and n > 1 else n // 2`; với `n = 1` biểu thức `1 // 2 = 0` cũng ra `0` nếu dùng nhánh chẵn, nên có thể viết `n // 2 if n % 2 == 0 or n == 1 else n`.
- Test nhanh bằng bảng `n = 1..8`: `0, 1, 3, 2, 5, 3, 7, 4`.

## 15. Biến thể của bài

1. **Số phần tối đa với `k` nhát cắt tự do (không cần bằng nhau):** bài toán "lazy caterer" cổ điển, `k(k+1)/2 + 1` cho đường thẳng không nhất thiết qua tâm.
2. **Chỉ cho phép đường kính:** khi đó `n` lẻ vô nghiệm; `n` chẵn cần `n/2` nhát.
3. **Chỉ cho phép bán kính:** luôn cần `n` nhát (với `n >= 2`).
4. **Chia hình tròn thành các phần có tỉ lệ cho trước:** cần các góc cộng lại 360 độ; xét số tia phân biệt.
5. **Bánh pizza với nhiều lớp:** thêm các nhát cắt ngang và đếm số mảnh.
6. **Cắt hình vuông thành `n` phần bằng nhau bằng các nhát thẳng đi qua tâm:** tương tự nhưng phải kiểm tra tính đối xứng.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử `k` tăng dần | đếm số phần cho mỗi `k` | `O(n)` | `O(1)` | để hiểu bài, kiểm tra công thức |
| 2. Phủ tia bằng `set` | xóa tia đối khi `n` chẵn | `O(n)`-`O(n^2)` | `O(n)` | khi muốn mô phỏng hình học |
| 3. Công thức rẽ nhánh | chẵn/lẻ và `n = 1` | `O(1)` | `O(1)` | cách nộp bài chuẩn |
| 4. Bảng tra cứu | dựng sẵn `n <= 100` | `O(1)` mỗi truy vấn | `O(100)` | nhiều truy vấn lặp lại |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().numberOfCuts(10)` bằng bao nhiêu? `Solution().numberOfCuts(9)`?

**Bài 2 (tính tay).** Với `n = 12`, vẽ các tia ở 0, 30, ..., 330 độ và đếm số đường kính cần cắt.

**Bài 3 (điền TODO).** Hoàn thành hàm `cuts_only_diameters(n)` trả về số nhát nếu **chỉ** được cắt bằng đường kính, và `-1` nếu bất khả thi.

In [26]:
def cuts_only_diameters(n):
    # TODO: n chẵn -> n // 2, n lẻ -> -1 (không thể)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `rays_angles(n)` trả về danh sách góc (độ) của các tia khi chia đều thành `n` phần (`n >= 2`).

**Bài 5 (biến thể / mini challenge).** Viết `total_cuts(ns)` nhận danh sách các `n` và trả về tổng số nhát cắt nếu mỗi chiếc bánh được cắt riêng theo đúng `n` của nó. Kiểm tra `total_cuts([1, 2, 3, 4]) == 0 + 1 + 3 + 2`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `10` chẵn nên `5`; `9` lẻ nên `9`. Bài 3: `n % 2 == 0` thì trả `n // 2`.
- **Mức 2 (vừa):** Bài 2: tia `i` có tia đối `i + 6`, nên có `6` cặp, đáp án `6`. Bài 4: `[i * 360 / n for i in range(n)]`.
- **Mức 3 (sâu):** Bài 5: dùng `sum(Solution().numberOfCuts(n) for n in ns)`. Kiểm tra ví dụ: `0 + 1 + 3 + 2 = 6`. Thử viết thêm một test với danh sách rỗng, tổng phải là `0`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert Solution().numberOfCuts(10) == 5
assert Solution().numberOfCuts(9) == 9

# Bài 2
assert Solution().numberOfCuts(12) == 6

# Bài 3
def cuts_only_diameters(n):
    return n // 2 if n % 2 == 0 else -1

# Bài 4
def rays_angles(n):
    return [i * 360 / n for i in range(n)]

# Bài 5
def total_cuts(ns):
    return sum(Solution().numberOfCuts(n) for n in ns)

assert cuts_only_diameters(8) == 4
assert cuts_only_diameters(7) == -1
assert cuts_only_diameters(2) == 1
assert rays_angles(4) == [0.0, 90.0, 180.0, 270.0]
assert rays_angles(3) == [0.0, 120.0, 240.0]
assert len(rays_angles(6)) == 6
assert total_cuts([1, 2, 3, 4]) == 6
assert total_cuts([]) == 0
assert total_cuts([100, 99]) == 50 + 99
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm số tia cách đều và ghép cặp tia đối)

```python
class Solution:
    def numberOfCuts(self, n: int) -> int:
        if n == 1:
            return 0
        return n // 2 if n % 2 == 0 else n
```

**Checklist**

- [ ] Đã xử lý `n = 1` trả về `0`?
- [ ] `n` chẵn trả về `n // 2`, `n` lẻ (>= 3) trả về `n`?
- [ ] Không nhầm "chẵn" với "chia hết cho 4"?
- [ ] Đã thử tay `n = 2, 3, 4, 6` và khớp ví dụ của đề (`4 -> 2`, `3 -> 3`)?
- [ ] Không dùng `(n + 1) // 2` hay `n // 2` cho mọi `n`?
- [ ] Signature đúng `numberOfCuts(self, n: int) -> int`?
- [ ] Giá trị trả về là số nguyên, không phải số thực (`/` thay vì `//`)?